# GPAW DFT HEA L12
Upload HEA_L12_SQS_32atoms.cif first.

In [ ]:
import os, tarfile, glob, shutil
!rm -rf /usr/local/share/gpaw setups.tar.gz /tmp/gpaw_extract
!apt-get install -y libxc-dev libblas-dev liblapack-dev libscalapack-mpi-dev gcc gfortran libopenmpi-dev > /dev/null
!pip install gpaw cupy-cuda12x ase > /dev/null
target_dir = '/usr/local/share/gpaw/setups'
os.makedirs(target_dir, exist_ok=True)
print('Downloading setups...')
!wget -q https://wiki.fysik.dtu.dk/gpaw-files/gpaw-setups-24.11.0.tar.gz -O setups.tar.gz
tmp_dir = '/tmp/gpaw_extract'
os.makedirs(tmp_dir, exist_ok=True)
with tarfile.open('setups.tar.gz') as tar:
    tar.extractall(tmp_dir)
!rm setups.tar.gz
found = glob.glob(f'{tmp_dir}/**/*.PBEsol', recursive=True)
if not found:
    found = glob.glob(f'{tmp_dir}/**/Al*', recursive=True)
if found:
    src = os.path.dirname(found[0])
    for f in os.listdir(src):
        shutil.copy2(os.path.join(src, f), os.path.join(target_dir, f))
    print(f'Copied {len(os.listdir(target_dir))} files.')
    print(f'Al files: {[f for f in os.listdir(target_dir) if f.startswith("Al")]}')
else:
    print('ERROR: no setups found')
os.environ['GPAW_SETUP_PATH'] = target_dir
from gpaw import GPAW, FermiDirac
import gpaw
print(f'GPAW {gpaw.__version__} ready')

In [ ]:
from ase.io import read, write
from ase.optimize import BFGS
try:
    from ase.filters import FrechetCellFilter as CellFilter
except ImportError:
    try:
        from ase.filters import ExpCellFilter as CellFilter
    except ImportError:
        from ase.constraints import ExpCellFilter as CellFilter
atoms = read('HEA_L12_SQS_32atoms.cif')
print(f'Atoms: {len(atoms)}')

# PBEsol setups are missing in the standard archive, using PBE instead
calc = GPAW(
    mode='lcao',
    basis='dzp',
    kpts=(4, 4, 4),
    xc='PBE',
    occupations=FermiDirac(width=0.02),
    txt='-'
)
atoms.calc = calc
print('Ready')

In [ ]:
e_filter = CellFilter(atoms)
opt = BFGS(e_filter)
print('Relaxing...')
opt.run(fmax=0.05)
print('Done')
print(f'Energy: {atoms.get_potential_energy():.4f} eV')
print(f'Cell: {atoms.cell}')
write('HEA_L12_relaxed.cif', atoms)
print('Saved HEA_L12_relaxed.cif')

0 GPU